**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Causal Inference

The question every regression dodges: what happens if we **intervene**? DAGs and d-separation, confounding and the backdoor adjustment, and the modern estimators — all on simulated worlds where we can *run the true intervention* and check the answer, the luxury real data never grants.

## 1. Pre-requisites

[Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb), [Independence](../Intro_Math/Analysis/Independence.ipynb); regression fluency ([Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
def ols(X, y):
    X1 = np.c_[np.ones(len(X)), X]
    return np.linalg.lstsq(X1, y, rcond=None)[0]

---
### 🕐 Session 1 of 3 — *Correlation, Confounding & the do-Operator* (~40 min)
**Goal:** watch a confounder manufacture a correlation; define intervention as graph surgery.
**Feeds into:** Session 2 (backdoor adjustment).

---

## 2. Seeing vs Doing

💡 **Intuition.** $P(Y \mid X = x)$ answers 'what do I expect of Y among units *observed* to have X = x?' — but observed X carries its causes with it. $P(Y \mid do(X = x))$ answers 'what if I *set* X to x?' — **graph surgery**: delete the arrows into X, because your intervention, not the world, chose it. The two differ exactly when a **confounder** feeds both X and Y. Our simulated world makes this concrete because we can *actually perform* the do — rerun the world with X forced.

In [ ]:
# World 1: exercise (X) → health (Y), both driven by age (Z). TRUE causal effect: +2.0
# the interventional ORACLE: force X and measure the response directly

# YOUR CODE HERE


**What just happened.** The regression says the effect of exercise on health is **5.40**. The truth — measured by actually forcing $X$ and rerunning the world — is **1.98**, against the 2.0 written into the generating code. The regression is off by a factor of **2.7**.

**And it is not off because of noise.** With $n = 20{,}000$ the standard error on that slope is roughly 0.02, so 5.40 sits about **170 standard errors** from the truth. **This is bias, and bias does not shrink with sample size.** Collect ten million points and you get 5.40 with tighter error bars: a beautifully precise measurement of the wrong quantity. That single sentence is the most important thing in this session, and it is what separates a statistical problem from a causal one.

**Decompose the 5.40 and the mechanism is visible.** Since $Y = 2X - 0.5Z + \ldots$ and $X = 8 - 0.1Z + \varepsilon$:

$$\frac{\mathrm{Cov}(X,Y)}{\mathrm{Var}(X)} = \frac{2\mathrm{Var}(X) - 0.5\mathrm{Cov}(X,Z)}{\mathrm{Var}(X)} \approx \underbrace{2.0}_{\text{causal}} + \underbrace{3.4}_{\text{backdoor path via age}}$$

Age pushes exercise *down* and health *down*, so the two negatives multiply to a positive contribution — the confounding **adds to** the true effect rather than cancelling it. The regression cannot tell the two terms apart; it reports their sum and calls it "the coefficient on X".

**Which is why the intervention is a different computation, not a better one.** Look at the line `if do_x is not None: X = np.full(n, float(do_x))`. It **overwrites the assignment mechanism** — age no longer determines exercise, because we do. That is $do(\cdot)$ implemented literally: delete the arrows into $X$, leave everything else alone. The remaining $Z \to Y$ arrow still operates, which is why the intervened world still has age affecting health; we severed one edge, not the variable.

**Note what this demo has that no real study does.** We can run both worlds. In practice you observe one, and the interventional quantity is **not identified by the data alone** — it requires assumptions about the graph, which is why causal claims are argued rather than computed. Simulated worlds are the only place to *learn* these methods, precisely because they are the only place the answer key exists.

**Finally, the practical shape of the error.** A naive analyst here would report that a unit of exercise buys 5.4 health points, and would be confident about it. Acting on that number — funding a programme sized to a 5.4 effect — buys 2.0. **The failure is not that the model fits badly; it fits the observational data perfectly.** It answers a question nobody asked.

---
### 🕐 Session 2 of 3 — *d-Separation & the Backdoor Adjustment* (~40 min)
**Goal:** block the right paths: adjust for confounders, DON'T adjust for colliders — both verified.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (modern estimators).

---

## 3. Which Variables to Control For

💡 **Intuition.** The graph answers it. Non-causal association flows along **backdoor paths** (X ← Z → Y); conditioning on Z *blocks* them — so regressing Y on X **and Z** recovers the causal slope. But the rule cuts both ways: a **collider** (X → C ← Y) is blocked *by default*, and conditioning on it **opens** a spurious path — 'controlling for everything' is how careful-sounding analyses create bias from thin air (selection on admission, hospitalization, employment...). d-separation is the complete bookkeeping of which paths are open.

In [ ]:
# Backdoor adjustment on World 1: add the confounder to the regression
# World 2: X and Y CAUSALLY UNRELATED, but both cause C (a collider)

# YOUR CODE HERE


**What just happened.** Two results that point in opposite directions, and the contrast is the session:

| world | analysis | result | truth |
|---|---|---|---|
| 1 | regress $Y$ on $X$ | 5.40 | 2.0 |
| 1 | regress $Y$ on $X$ **and $Z$** | **2.014** | 2.0 |
| 2 | regress $Y$ on $X$ | −0.002 | 0 |
| 2 | regress $Y$ on $X$ **and $C$** | **−0.800** | 0 |
| 2 | regress on $X$ among $C > 1$ | **−0.490** | 0 |

**Adding a variable fixed the first world and broke the second.** Same action, opposite consequences — which is why "control for everything you measured" is not a conservative default but a way to manufacture bias from nothing.

**World 1 is the expected half.** Conditioning on age blocks the backdoor path $X \leftarrow Z \rightarrow Y$, leaving only the causal path, and 2.014 against a truth of 2.0 is within noise. Confounding removed.

**World 2 is the half that overturns the instinct, and the number is exactly predictable.** $X$ and $Y$ are drawn **independently**; there is no causal path between them at all. Yet controlling for $C = X + Y + 0.5\varepsilon$ produces $-0.800$, and that is not sampling noise — solving the normal equations with $\mathrm{Var}(C) = 2.25$ and $\mathrm{Cov}(X,C) = \mathrm{Cov}(Y,C) = 1$ gives $-1/1.25 = -0.8$ exactly. **The bias is the population value.** More data reproduces it more precisely.

**The mechanism, in one sentence you can say aloud.** $C$ is "got admitted" and $X + Y$ is "talent plus luck". Among people who got in, learning that someone has high talent tells you their luck must have been low — *something* had to get them in. **Conditioning on a common effect makes its independent causes dependent.** The negative sign is not incidental: it is the arithmetic of a fixed budget being split.

**The third row is where students will actually meet this, and it involves no regression at all.** Restricting to $C > 1$ gives $-0.490$. **Selecting your sample on a collider *is* conditioning on it.** Study only hospitalised patients, only admitted applicants, only employed workers, only firms that survived — and the bias arrives with the dataset, before any modelling decision is made. Nobody typed the word "control".

**Which yields the rule this session exists to establish.** Adjust for confounders; never adjust for colliders or their descendants. And note the uncomfortable part: **$Z$ and $C$ are statistically indistinguishable in the data.** Both correlate with $X$ and with $Y$. No variable-selection procedure, no significance test, no regularisation path can tell you which is which — only a claim about the *causal structure* can, and that claim comes from domain knowledge, not from the sample.

**Finally, note the general machinery these two cases are instances of.** d-separation says a path is blocked or open by three rules — chain, fork, collider — with conditioning *closing* the first two and *opening* the third. $X \perp Y \mid S$ exactly when every path between them is blocked. Both worlds above are one-line applications of that bookkeeping, and it scales to graphs far too large to reason about by story.

---
### 🕐 Session 3 of 3 — *Modern Estimators* (~40 min)
**Goal:** IPW, standardization, doubly-robust — three routes to the same do, audited against the oracle.
**Builds on:** Session 2.

---

## 4. Three Roads to the Interventional Answer

💡 **Intuition.** With binary treatment, three standard estimators of $E[Y|do(T{=}1)] - E[Y|do(T{=}0)]$:
1. **Standardization**: model $E[Y|T,Z]$, average over the *whole* Z population.
2. **Inverse propensity weighting (IPW)**: reweight each unit by $1/P(T{=}t|Z)$ — manufacture the randomized trial that nature refused to run.
3. **Doubly robust**: combine both; consistent if *either* model is right — insurance against your own misspecification.
The oracle discipline continues: we simulate the true counterfactuals and grade all three.

In [ ]:
# binary-treatment world: treatment probability depends on Z; effect heterogeneous
# 1) standardization (outcome regression with interaction)
# 2) IPW with a logistic propensity (fit by Newton in 6 lines)
# 3) doubly robust (AIPW)

# YOUR CODE HERE


**What just happened.** Four estimates against a measured oracle of **+1.502** (theory says exactly 1.5, since $E[1.5 - 0.5Z] = 1.5$ for $E[Z]=0$):

| estimator | estimate | error |
|---|---|---|
| naive difference in means | +3.341 | **+1.84** |
| standardization | +1.490 | −0.012 |
| IPW | +1.498 | −0.004 |
| doubly robust | +1.498 | −0.004 |

**The naive estimate is 2.2× the truth, and the mechanism is the same as Session 1 with a binary treatment.** High-$Z$ units are both more likely to be treated ($p = \sigma(1.5Z)$) *and* have higher $Y$ regardless of treatment ($+2.0Z$). So the treated group is systematically different from the untreated one before any treatment occurs, and the difference in means measures that gap plus the effect.

**Three routes fix it, and they are genuinely different routes — which is the point.** Standardization fits the outcome model and averages it over **everyone's** $Z$, simulating both worlds. IPW never models the outcome at all; it reweights each unit by $1/P(T \mid Z)$, upweighting units that got the treatment they were unlikely to get, **manufacturing the randomised trial nature refused to run**. Doubly robust combines them and is consistent if *either* model is correct.

**Do not read the tiny differences as a ranking.** The spread between 1.490 and 1.498 is 0.008, well inside Monte Carlo noise at $n = 40{,}000$ — rerun with another seed and the ordering shuffles. All three agree with the oracle; none is demonstrated better than the others here.

**And note what this demo structurally cannot show.** Both the outcome model (linear with a $T{\times}Z$ interaction, exactly matching the truth) and the propensity model (logistic in $Z$, exactly matching the truth) are **correctly specified**. Doubly robust is designed to survive when one of them is wrong — so a simulation where both are right cannot exhibit its advantage. **The estimator's headline property is untested here.** The experiment worth running: drop the interaction term from the outcome model, or fit the propensity without $Z^2$ when the truth needs it, and watch standardization or IPW break while DR holds.

**One practical hazard is visible in the propensity values.** At $Z = 3$, $p \approx 0.989$, so an untreated unit there carries weight $1/(1-p) \approx 90$. A handful of such units can dominate the IPW average. Here $Z$ is standard normal, the tails are thin, and it behaves — but with stronger confounding or heavier tails **IPW becomes extremely high-variance**, which is why weight trimming and overlap diagnostics are standard practice. Standardization has no analogous blow-up; it has the opposite weakness, relying entirely on the outcome model extrapolating correctly.

**Finally, the boundary that no estimator crosses.** All three assumed **no unmeasured confounding** — that $Z$ captures everything feeding both $T$ and $Y$. Add a hidden variable and every number above moves, while **the observed data looks identical**. No amount of cleverness detects it. The graph tells you what to adjust for; only the *design* tells you whether you measured enough — which is why a randomised trial outranks any estimator: it severs the arrow into $T$ by construction rather than by assumption.

**The honest boundary:** every method above assumed *no unmeasured confounding* — an assumption the data can never certify (that's what makes causal inference hard, and what instrumental variables, sensitivity analysis, and RCTs exist for). The graph tells you what to adjust; only design tells you whether you measured enough.

## 5. Conclusion

Seeing ≠ doing (naive slope 2× the truth, measured); backdoors are blocked by adjustment while colliders are *opened* by it (both manufactured on demand); and standardization/IPW/DR all recover the interventional oracle within noise. Regression answers questions about the world as it is; these tools answer questions about worlds we might make.

---
## Where next

- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — the statistical machinery under each estimator.
- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — predictions under distribution shift: causality's sibling problem.